In [1]:
import os
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
data_path = '/content/drive/MyDrive/Final Year Project/Data/Processed'


In [5]:
train_path = os.path.join(data_path, 'train')
test_path = os.path.join(data_path, 'test')
val_path = os.path.join(data_path, 'val')
reference_path = os.path.join('/content/drive/MyDrive/Final Year Project/Data/Processed', 'reference.csv')



In [6]:
ref_data = pd.read_csv(reference_path)

In [7]:
def get_label(video_id):
  if isinstance(video_id, str):
    # Extract the ID part from filename (e.g., '58031 (1).npy' -> '58031')
    clean_id = video_id.split('.')[0].split(' ')[0]
    try:
      video_id = int(clean_id)
    except ValueError:
      pass

  return ref_data[ref_data['video_id'] == video_id]['gloss'].values[0]

In [8]:
def load_file(file_path):
    """Load a single file and return (data, label)."""
    X_data = np.load(file_path)
    label = get_label(os.path.basename(file_path))
    return X_data, label

In [9]:
def load_data(path):
    """Load all .npy files in parallel with a progress bar."""
    X, y = [], []
    files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.npy')]

    with ThreadPoolExecutor() as executor:
        futures = {executor.submit(load_file, f): f for f in files}

        # Wrap as_completed with tqdm to show progress
        for future in tqdm(as_completed(futures), total=len(futures), desc="Loading Files"):
            data, label = future.result()
            X.append(data)
            y.append(label)

    print(f"{len(y)} records loaded from {path}")
    return np.array(X), np.array(y)

In [10]:
X_train, y_train = load_data(train_path)
X_test, y_test = load_data(test_path)
X_val, y_val = load_data(val_path)

Loading Files:   0%|          | 0/8724 [00:00<?, ?it/s]

8724 records loaded from /content/drive/MyDrive/Final Year Project/Data/Processed/train


Loading Files:   0%|          | 0/1473 [00:00<?, ?it/s]

1473 records loaded from /content/drive/MyDrive/Final Year Project/Data/Processed/test


Loading Files:   0%|          | 0/2348 [00:00<?, ?it/s]

2348 records loaded from /content/drive/MyDrive/Final Year Project/Data/Processed/val


In [11]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Combine all labels to ensure consistent encoding across train, test, and val sets
all_labels = np.concatenate((y_train, y_test, y_val))

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit on all labels and transform
label_encoder.fit(all_labels)
y_train_encoded = label_encoder.transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
y_val_encoded = label_encoder.transform(y_val)

# Determine the total number of unique classes from all_labels
num_classes = len(label_encoder.classes_)

num_classes

2000

In [12]:
# Convert integer labels to one-hot encoded vectors, explicitly providing num_classes
y_train_one_hot = to_categorical(y_train_encoded, num_classes=num_classes)
y_test_one_hot = to_categorical(y_test_encoded, num_classes=num_classes)
y_val_one_hot = to_categorical(y_val_encoded, num_classes=num_classes)

# Reshape the data to flatten the landmark coordinates (92 * 3 = 276 features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], -1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], -1)
X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], -1)

# Determine the input shape for the LSTM layer
# X_train shape is now (num_samples, timesteps, features)
input_shape = (X_train.shape[1], X_train.shape[2])


print(f"Input shape for LSTM: {input_shape}")
print(f"Shape of y_train_one_hot: {y_train_one_hot.shape}")
print(f"Shape of y_test_one_hot: {y_test_one_hot.shape}")
print(f"Shape of y_val_one_hot: {y_val_one_hot.shape}")

Input shape for LSTM: (30, 276)
Shape of y_train_one_hot: (8724, 2000)
Shape of y_test_one_hot: (1473, 2000)
Shape of y_val_one_hot: (2348, 2000)


In [13]:

from tensorflow.keras.callbacks import TensorBoard

log_dir = os.path.join(data_path, 'logs')
tensorboard_callback = TensorBoard(log_dir=log_dir)

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Build the LSTM model
model = Sequential()
model.add(LSTM(128, return_sequences=True, activation='relu', input_shape=input_shape)) # Increased units
model.add(Dropout(0.2)) # Added Dropout
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(Dropout(0.2)) # Added Dropout
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dropout(0.2)) # Added Dropout
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [15]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [16]:
# Display model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 128)        │       207,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 30, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2000)           │        66,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 460,592 (1.76 MB)

 Trainable params: 460,592 (1.76 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [18]:
history = model.fit(
    X_train, y_train_one_hot,
    validation_data=(X_val, y_val_one_hot),
    epochs=200, # Increased epochs
    callbacks=[early_stop]
)

Epoch 1/200
273/273 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - accuracy: 6.8776e-04 - loss: 7.6037 - val_accuracy: 0.0017 - val_loss: 7.5040
Epoch 2/200
273/273 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.0017 - loss: 7.3384 - val_accuracy: 0.0021 - val_loss: 7.2695
Epoch 3/200
273/273 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.0021 - loss: 7.1029 - val_accuracy: 0.0017 - val_loss: 7.2015
Epoch 4/200
273/273 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.0025 - loss: 7.0088 - val_accuracy: 8.5179e-04 - val_loss: 7.2187
Epoch 5/200
273/273 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.0026 - loss: 6.9497 - val_accuracy: 8.5179e-04 - val_loss: 7.1920
Epoch 6/200
273/273 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.0026 - loss: 6.9165 - val_accuracy: 8.5179e-04 - val_loss: 7.2392
Epoch 7/200
273/273 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.0021 - loss: 6.8900 - val_accuracy: 0.0013 - val_loss: 7.2568
Epoch 8/200
273/273 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.0032 

In [19]:
loss, accuracy = model.evaluate(X_test, y_test_one_hot, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test Loss: 7.1623
Test Accuracy: 0.0027


In [21]:
model.predict(X_train[0])

ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("data:0", shape=(30, 276), dtype=float32) with name 'keras_tensor' and path ''. Expected shape (None, 30, 276), but input has incompatible shape (30, 276)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(30, 276), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [ ]:
X_train.shape

In [ ]:
import numpy as np

# Take a single sample from X_test for prediction
single_sample = np.expand_dims(X_test[0], axis=0)

# Make a prediction
prediction = model.predict(single_sample)

# Get the predicted class index
predicted_class_index = np.argmax(prediction)

# Get the actual label using the label_encoder
predicted_label = label_encoder.inverse_transform([predicted_class_index])[0]

# Get the true label for comparison
true_label = y_test[0]

print(f"Predicted label: {predicted_label}")
print(f"True label: {true_label}")

In [ ]:
y_val_one_hot.shape

In [ ]:
y_train

In [ ]:
#save model
model.save('sign_language.h5')